<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/audio_batch_normalizer_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📀 Audio Normalization & Export Tool for 3 MP3 Files
This notebook:
- Installs dependencies
- Uploads 3 MP3 files
- Normalizes RMS between -23dB and -18dB
- Ensures peak ≤ -3dB
- Exports MP3 at 192kbps+, 44.1kHz, CBR
- Provides download links

In [ ]:
# 🔧 Install dependencies
!pip install pydub ffmpeg-normalize

In [ ]:
# 📚 Import libraries
from pydub import AudioSegment
from google.colab import files
import os

In [ ]:
# ⚙️ Define the processing function
def process_audio(file, target_rms=-20.0):
    audio = AudioSegment.from_file(file)
    audio = audio.set_channels(2)  # Convert to stereo

    change_in_db = target_rms - audio.dBFS
    audio = audio.apply_gain(change_in_db)

    peak = audio.max_dBFS
    if peak > -3.0:
        audio = audio.apply_gain(-3.0 - peak)

    audio = audio.set_frame_rate(44100)
    output_filename = f"adjusted_{os.path.splitext(file)[0]}.mp3"
    audio.export(output_filename, format="mp3", bitrate="192k", parameters=["-acodec", "libmp3lame", "-b:a", "192k", "-ar", "44100", "-ac", "2"])
    return output_filename

In [ ]:
# 📤 Upload 3 MP3 files
print("Upload 3 MP3 files...")
uploaded = files.upload()

In [ ]:
# ▶️ Process and download each file
for fname in uploaded.keys():
    if fname.lower().endswith(('.mp3', '.m4a', '.wav', '.flac')):
        out = process_audio(fname)
        print(f"✅ Processed {fname} -> {out}")
        files.download(out)
    else:
        print(f"Unsupported file type: {fname}")